In [ ]:
# Importar las librerías necesarias
import pandas as pd
import sqlite3

In [ ]:
# Conexión a la base de datos SQLite
conn = sqlite3.connect('test_database.db')
cur = conn.cursor()

In [ ]:
# Comprobar y listar las tablas disponibles en la base de datos
cur. execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cur.fetchall())

[('Customer',), ('Product',), ('Sale',), ('Store',), ('Storage',), ('Provider',)]


In [ ]:
# Definir una función para ejecutar consultas y mostrar los resultados en formato DataFrame
def read_df(cur,command):
    result = cur.execute(command)
    col_names = list(map(lambda x: x[0], cur.description))
    df_query = pd.DataFrame(result, columns =col_names)
    print(df_query) 

In [ ]:
# Mostrar productos que NO son suministrados por el proveedor con ID = 3
read_df(cur,"""SELECT ProductID, ProductName
               FROM Product
               EXCEPT
                    SELECT Product.ProductID, Product.ProductName
                    FROM Product 
                    INNER JOIN Provider ON Provider.ProviderID = Product.ProviderID
                    WHERE Provider.ProviderID = 3;""")

    ProductID ProductName
0           2    Democrat
1           3        Part
2           4   Represent
3           5      Single
4           6      Lawyer
..        ...         ...
63         92      Agency
64         93     However
65         95       Clear
66         96   Recognize
67         98         How

[68 rows x 2 columns]


In [ ]:
# Mostrar productos que NO son suministrados por el proveedor con ID = 3 usando NOT IN
read_df(cur,"""SELECT ProductID, ProductName
               FROM Product
               WHERE ProductID NOT IN
                    (SELECT Product.ProductID
                    FROM Product 
                    INNER JOIN Provider ON Provider.ProviderID = Product.ProviderID
                    WHERE Provider.ProviderID = 3);""")

    ProductID ProductName
0           2    Democrat
1           3        Part
2           4   Represent
3           5      Single
4           6      Lawyer
..        ...         ...
63         92      Agency
64         93     However
65         95       Clear
66         96   Recognize
67         98         How

[68 rows x 2 columns]


In [ ]:
# Obtener el cliente que gastó más dinero
read_df(cur,"""SELECT FirstName, LastName, SUM(Price*Quantity) AS Monto 
               FROM Customer 
               LEFT JOIN Sale
               USING (CustomerID)
               LEFT JOIN Product
               USING (ProductID)
               GROUP BY FirstName, LastName
               HAVING SUM(Price*Quantity)=(
                    SELECT MAX(Monto)
                    FROM(
                         SELECT SUM(Price*Quantity) AS Monto 
                         FROM Customer
                         LEFT JOIN Sale
                         USING (CustomerID)
                         LEFT JOIN Product
                         USING (ProductID)
                         GROUP BY FirstName, LastName));""")

  FirstName LastName     Monto
0    Brandy    Woods  190265.2


In [ ]:
# Obtener el proveedor con más ventas
read_df(cur,"""SELECT Provider.ProviderName, COUNT(SaleID) AS Q
               FROM Provider
               LEFT JOIN Product
               USING (ProviderID)
               LEFT JOIN Sale 
               USING (ProductID)
               GROUP BY Provider.ProviderName
               ORDER BY Q DESC;""")

     ProviderName    Q
0       Moyer LLC  354
1  Maxwell-Carter  329
2     Parsons LLC  317


In [ ]:
# Encontrar los meses con ventas por debajo del promedio
read_df(cur,"""SELECT strftime('%m', SaleDate) AS MONTH, SUM(Price * Quantity) AS MONTO
               FROM Sale
               LEFT JOIN Product
               USING (ProductID)
               GROUP BY strftime('%m', SaleDate)
               HAVING SUM(Price * Quantity)<(
                    SELECT AVG(MONTO)
                    FROM(
                        SELECT SUM(Price * Quantity) AS MONTO
                        FROM Sale
                        LEFT JOIN Product
                        USING (ProductID) 
                        GROUP BY strftime('%m', SaleDate)));""")

  MONTH      MONTO
0    01  421607.24
1    02  400407.21
2    06  309349.31
